# 🌙 07. SuperPoint Deep Feature & Keypoint Extraction

**Mission Context**: Deep sub-pixel keypoint detection resilient to extreme lunar shadow migration, craters, and scale variation.  
**Objectives**:
- Extract SuperPoint keypoint coordinates $\mathbf{x} \in \mathbb{R}^2$, response confidence scores, and dense 256-D descriptors $\mathbf{d} \in \mathbb{R}^{256}$.
- Measure keypoint density ($K / \text{pixel}$) and spatial coverage.
- Export `features_source.npz` and `features_reference.npz`.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.features import SuperPointExtractor
from lunar_core.synthetic_data import LunarSyntheticGenerator

config = load_config()
gen = LunarSyntheticGenerator(size=(512, 512), seed=42)
pair = gen.generate_registered_pair(rotation_deg=10.0, scale=1.04, tx=25.0, ty=-15.0)

ref_img = pair["reference_image"]
src_img = pair["source_image"]

sp = SuperPointExtractor(max_keypoints=1024, keypoint_threshold=0.005)
feats_ref = sp.extract(ref_img)
feats_src = sp.extract(src_img)

print(f"Reference Keypoints Extracted: {len(feats_ref['keypoints'])}")
print(f"Source Keypoints Extracted: {len(feats_src['keypoints'])}")


In [ ]:
# Visualize SuperPoint Keypoints
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Reference keypoints
ax_ref = cv2.cvtColor(ref_img, cv2.COLOR_GRAY2BGR)
for pt in feats_ref['keypoints']:
    cv2.circle(ax_ref, (int(pt[0]), int(pt[1])), 2, (0, 255, 0), -1)

axes[0].imshow(cv2.cvtColor(ax_ref, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Reference Frame SuperPoint Keypoints (N={len(feats_ref['keypoints'])})", fontweight='bold')
axes[0].axis('off')

# Source keypoints
ax_src = cv2.cvtColor(src_img, cv2.COLOR_GRAY2BGR)
for pt in feats_src['keypoints']:
    cv2.circle(ax_src, (int(pt[0]), int(pt[1])), 2, (0, 0, 255), -1)

axes[1].imshow(cv2.cvtColor(ax_src, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Source Frame SuperPoint Keypoints (N={len(feats_src['keypoints'])})", fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/07_superpoint_extractions.png", dpi=300)
plt.show()


In [ ]:
# Export NPZ feature files
os.makedirs("outputs/features", exist_ok=True)
np.savez_compressed(
    "outputs/features/features_reference.npz",
    keypoints=feats_ref["keypoints"],
    scores=feats_ref["scores"],
    descriptors=feats_ref["descriptors"]
)

np.savez_compressed(
    "outputs/features/features_source.npz",
    keypoints=feats_src["keypoints"],
    scores=feats_src["scores"],
    descriptors=feats_src["descriptors"]
)

print("Exported outputs/features/features_reference.npz and features_source.npz")
